# 🎬 YouTube Shorts ワンクリック自動生成

---

## ✏️ 毎回やること（セル1だけ変える）

| 設定項目 | 説明 |
|---|---|
| `YOUTUBE_API_KEY` | **初回だけ**入力。Google Cloud Console で取得（**無料**） |
| `CLAUDE_API_KEY` | **初回だけ**入力。console.anthropic.com で取得（有料・`sk-ant-`で始まる） |
| `PEXELS_API_KEY` | **初回だけ**入力。pexels.com/api で取得（**無料**） |
| `THEME` | **毎回**テーマを書き換える |
| `VIDEO_COUNT` | 生成する本数（1〜10） |

あとは「**ランタイム → すべてのセルを実行**」をクリックするだけ！

---

## 💰 Claude APIの費用目安
| 使い方 | 費用 |
|---|---|
| 1回10本生成 | 約5円 |
| 毎日10本 × 30日 | 約160円/月 |

---

## 🎨 画像スタイル
| スタイル名 | 見た目 | 向いているテーマ |
|---|---|---|
| `realistic` | 写真そのまま | 筋トレ・料理・ビジネス |
| `anime` | アニメ風 | 恋愛・感情・エンタメ |
| `manga` | 漫画風（白黒） | 怖い話・歴史・雑学 |
| `illustration` | イラスト風 | 子ども向け・ライフスタイル |

---

## ⏱ 目安時間
| 本数 | 目安 |
|---|---|
| 1本 | 約5〜10分 |
| 5本 | 約25〜40分 |
| 10本 | 約50〜80分 |

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  セル1: ここだけ変える（毎回）                        ║
# ╚══════════════════════════════════════════════════════╝

# ── APIキー ──────────────────────────────────────────────
YOUTUBE_API_KEY = ''   # ← Google Cloud Console で取得（無料）
CLAUDE_API_KEY  = ''   # ← console.anthropic.com で取得（sk-ant-で始まる）
PEXELS_API_KEY  = ''   # ← pexels.com/api で取得（無料）

# ── テーマ（毎回変える） ─────────────────────────────────
THEME = 'ダイエット'
#  例: 'ダイエット' / '筋トレ' / 'NISA' / '投資' / '英語学習' / '副業'

# ── 生成する動画の本数 ───────────────────────────────────
VIDEO_COUNT = 10        # ← 1〜10本

# ── 動画の長さ ───────────────────────────────────────────
DURATION = 45           # ← 秒数（30〜60）

# ── 画像スタイル ─────────────────────────────────────────
IMAGE_STYLE = 'realistic'    # 写真そのまま
# IMAGE_STYLE = 'anime'        # アニメ風
# IMAGE_STYLE = 'manga'        # 漫画風（白黒）
# IMAGE_STYLE = 'illustration' # イラスト風

# ╔══════════════════════════════════════════════════════╗
# ║  ↑ 変えるのはここまで。以下は触らなくてOK            ║
# ╚══════════════════════════════════════════════════════╝

if not YOUTUBE_API_KEY.strip():
    raise ValueError('❌ YOUTUBE_API_KEY を入力してください（Google Cloud Console で取得）')
if not CLAUDE_API_KEY.strip():
    raise ValueError('❌ CLAUDE_API_KEY を入力してください（console.anthropic.com で取得）')
if not PEXELS_API_KEY.strip():
    raise ValueError('❌ PEXELS_API_KEY を入力してください（pexels.com/api で取得）')

import os
os.makedirs('/content/output', exist_ok=True)

print('設定内容を確認します...')
print(f'  テーマ      : {THEME}')
print(f'  生成本数    : {VIDEO_COUNT}本')
print(f'  動画の長さ  : {DURATION}秒')
print(f'  画像スタイル: {IMAGE_STYLE}')
print()
print('✅ 設定完了！')

In [ ]:
# 【自動】必要なツールをインストール（触らなくてOK）
!pip install -q gtts requests opencv-python-headless
!apt-get install -q -y ffmpeg fonts-noto-cjk
print('✅ ツールのインストール完了')

In [ ]:
# 【自動】① YouTubeトレンド取得 → ② 人気動画の詳細分析 → ③ Claude AI分析 → ④ 切り口生成

import requests as _req
import json, re

def fetch_youtube_trends(theme):
    """人気動画を検索して動画IDを取得"""
    url = 'https://www.googleapis.com/youtube/v3/search'
    params = {
        'part': 'snippet',
        'q': f'{theme} shorts',
        'type': 'video',
        'order': 'viewCount',
        'regionCode': 'JP',
        'relevanceLanguage': 'ja',
        'maxResults': 10,
        'key': YOUTUBE_API_KEY,
        'videoDuration': 'short',
    }
    try:
        r = _req.get(url, params=params, timeout=15)
        if r.status_code == 200:
            items = r.json().get('items', [])
            return [{'id': item['id'].get('videoId',''),
                     'title': item['snippet'].get('title',''),
                     'channel': item['snippet'].get('channelTitle','')} for item in items
                    if item['id'].get('videoId')]
        else:
            print(f'  ⚠ YouTube検索API: {r.status_code}')
            return []
    except Exception as e:
        print(f'  ⚠ YouTube API失敗: {e}')
        return []

def fetch_video_stats(video_ids):
    """動画IDから再生数・いいね数・タグ・説明文を取得"""
    if not video_ids: return []
    url = 'https://www.googleapis.com/youtube/v3/videos'
    params = {
        'part': 'snippet,statistics',
        'id': ','.join(video_ids[:10]),
        'key': YOUTUBE_API_KEY,
    }
    try:
        r = _req.get(url, params=params, timeout=15)
        if r.status_code == 200:
            results = []
            for item in r.json().get('items', []):
                snip = item.get('snippet', {})
                stats = item.get('statistics', {})
                results.append({
                    'title':       snip.get('title', ''),
                    'description': snip.get('description', '')[:200],
                    'tags':        snip.get('tags', [])[:8],
                    'views':       int(stats.get('viewCount', 0)),
                    'likes':       int(stats.get('likeCount', 0)),
                    'comments':    int(stats.get('commentCount', 0)),
                })
            # 再生数順にソート
            return sorted(results, key=lambda x: x['views'], reverse=True)
        else:
            print(f'  ⚠ YouTube統計API: {r.status_code}')
            return []
    except Exception as e:
        print(f'  ⚠ 統計取得失敗: {e}')
        return []

CLAUDE_URL = 'https://api.anthropic.com/v1/messages'

def call_ai(prompt, tokens=4096):
    res = _req.post(
        CLAUDE_URL,
        headers={'x-api-key': CLAUDE_API_KEY,
                 'anthropic-version': '2023-06-01',
                 'content-type': 'application/json'},
        json={'model': 'claude-haiku-4-5-20251001',
              'max_tokens': tokens,
              'messages': [{'role': 'user', 'content': prompt}]},
        timeout=120
    )
    if res.status_code != 200:
        err = res.text[:300].replace(CLAUDE_API_KEY, '***')
        raise RuntimeError(f'Claude APIエラー ({res.status_code}): {err}')
    return res.json()['content'][0]['text']

def clean_title(t):
    t = re.sub(r'^#+\s*', '', t)
    t = re.sub(r'\*+', '', t)
    t = re.sub(r'^\d+[\.\)]\s*', '', t)
    return t.strip()

# ── ① YouTube人気動画を検索 ──────────────────────────────────
print(f'📺 YouTubeで「{THEME}」の人気動画を検索中...')
yt_basic = fetch_youtube_trends(THEME)

if yt_basic:
    print(f'  ✓ {len(yt_basic)}件の動画を発見')
    video_ids = [v['id'] for v in yt_basic if v['id']]

    # ── ② 再生数・いいね数・タグを詳細取得 ──────────────────
    print(f'  📊 動画の詳細データを取得中...')
    yt_stats = fetch_video_stats(video_ids)

    if yt_stats:
        print(f'  ✓ 統計データ取得完了（上位{len(yt_stats)}本）')
        print(f'\n  🔥 最も再生されている動画:')
        for v in yt_stats[:3]:
            views_str = f"{v['views']:,}" if v['views'] > 0 else '非公開'
            print(f'    再生:{views_str} │ {v["title"][:40]}')

        # 分析用テキスト（再生数・タグ・説明文を含む）
        analysis_lines = []
        for i, v in enumerate(yt_stats[:8]):
            tags_str = '・'.join(v['tags'][:5]) if v['tags'] else 'タグなし'
            analysis_lines.append(
                f'{i+1}. 「{v["title"]}」\n'
                f'   再生:{v["views"]:,} / いいね:{v["likes"]:,}\n'
                f'   タグ: {tags_str}\n'
                f'   説明: {v["description"][:80]}'
            )
        yt_data_text = '\n'.join(analysis_lines)
    else:
        yt_data_text = '\n'.join([f'{i+1}. 「{v["title"]}」' for i, v in enumerate(yt_basic)])
else:
    print('  ⚠ YouTube API取得できず。AI知識でトレンド分析します。')
    yt_data_text = f'テーマ「{THEME}」の一般的なトレンド情報'

# ── ③ Claude でトレンド＆音声スタイルを分析 ─────────────────
print(f'\n🤖 Claude AIでトレンドと音声スタイルを分析中...')
trend = call_ai(
    f'YouTubeの「{THEME}」人気Shortsデータ:\n{yt_data_text}\n\n'
    f'このデータを分析して以下をまとめてください:\n'
    f'1. バズっている動画の共通タイトルパターン（数字・煽り系キーワード）\n'
    f'2. 最初の3秒で使うべき強烈なフック\n'
    f'3. 視聴者が最後まで見る構成の特徴\n'
    f'4. 【音声スタイル】話し方のテンポ・リズム・語尾の特徴\n'
    f'   （例: テンポ速め・断定口調・「〜だよ！」系など）\n'
    f'箇条書きで簡潔に。'
)
print('  ✓ トレンド・音声分析完了')
print(f'\n  分析結果の一部:\n  {trend[:200]}...')

# ── ④ VIDEO_COUNT個の切り口を生成 ────────────────────────────
print(f'\n💡 切り口を{VIDEO_COUNT}個考案中...')
angles_raw = call_ai(
    f'テーマ「{THEME}」のYouTube Shortsタイトルを{VIDEO_COUNT}個出力してください。\n'
    f'トレンド分析: {trend[:500]}\n\n'
    f'【絶対守ること】\n'
    f'・タイトルのみ出力（説明・番号・記号なし）\n'
    f'・1行1タイトル\n'
    f'・数字を入れる（例: 3つの方法、1週間で）\n'
    f'・煽り系ワード（知らないと損、衝撃、やばい、本当は）\n'
    f'・20文字以内\n\n'
    f'出力例:\n痩せない本当の理由3つ\n1週間で-3kgの食事法\n食べても太らない時間帯',
    tokens=1024
)
ANGLES = [clean_title(l) for l in angles_raw.strip().split('\n')
          if l.strip() and len(l.strip()) > 3][:VIDEO_COUNT]
while len(ANGLES) < VIDEO_COUNT:
    ANGLES.append(f'{THEME}の秘密 Vol.{len(ANGLES)+1}')

# トレンド分析結果をセル4で使えるようにグローバル変数に保存
TREND_ANALYSIS = trend

print(f'\n生成する{VIDEO_COUNT}本:')
for i, a in enumerate(ANGLES):
    print(f'  {i+1:2d}. {a}')
print(f'\n✅ 切り口の決定完了')

In [ ]:
# 【自動】③〜⑥ 全動画を一括生成（触らなくてOK）

import cv2, numpy as np, subprocess, tempfile, time, wave
from pathlib import Path
from gtts import gTTS

def anime(img):
    c = cv2.bilateralFilter(cv2.bilateralFilter(img,9,250,250),9,250,250)
    g = cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),5)
    e = cv2.cvtColor(cv2.adaptiveThreshold(g,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,9,5),cv2.COLOR_GRAY2BGR)
    r = cv2.bitwise_and(c,e)
    h = cv2.cvtColor(r,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1] = np.clip(h[:,:,1]*1.5,0,255)
    return cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)

def manga(img):
    g = cv2.createCLAHE(2.0,(8,8)).apply(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY))
    e = cv2.dilate(cv2.Canny(cv2.GaussianBlur(g,(3,3),0),30,100),np.ones((2,2),np.uint8))
    _,t = cv2.threshold(g,180,255,cv2.THRESH_BINARY)
    return cv2.cvtColor(cv2.addWeighted(t,.75,cv2.bitwise_not(e),.25,0),cv2.COLOR_GRAY2BGR)

def illust(img):
    c = cv2.bilateralFilter(img,15,80,80)
    h = cv2.cvtColor(c,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1]=np.clip(h[:,:,1]*1.7,0,255); h[:,:,2]=np.clip(h[:,:,2]*1.1,0,255)
    c = cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)
    e = cv2.cvtColor(cv2.adaptiveThreshold(cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),7),255,
        cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,11,9),cv2.COLOR_GRAY2BGR)
    return cv2.bitwise_and(c,e)

STYLES = {'anime':anime,'manga':manga,'illustration':illust}

def ff(*a):
    r = subprocess.run(['ffmpeg','-y',*[str(x) for x in a]],
                       capture_output=True,text=True,timeout=300)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-800:])

def at(s):
    return f'{int(s//3600)}:{int((s%3600)//60):02d}:{int(s%60):02d}.{int((s%1)*100):02d}'

def get_wav_dur(path):
    try:
        with wave.open(str(path),'r') as wf:
            return wf.getnframes()/wf.getframerate()
    except:
        return 3.0

def clean_text(t):
    t = re.sub(r'\[速く\]|\[ゆっくり\]|\[強調\]','',t)
    t = re.sub(r'\[間\d+\.?\d*\]','、',t)
    t = re.sub(r'（ここにセリフ）|\(ここにセリフ\)|シーン\d+[（(][^)）]*[)）]\s*[:：]','',t)
    t = re.sub(r'^#+\s*','',t); t = re.sub(r'\*+','',t)
    return re.sub(r'\s+',' ',t).strip()

def wrap_subtitle(text, max_chars=13):
    text = text.strip()
    if len(text) <= max_chars:
        return text
    for i in range(max_chars, min(max_chars+6, len(text))):
        if i < len(text) and text[i] in '、。！？はがをにでもよね':
            return text[:i+1] + r'\N' + text[i+1:]
    return text[:max_chars] + r'\N' + text[max_chars:]

# シーンごとにクロップ位置を変えて動きを演出
CROP_OFFSETS = [
    (0, 0), (int(1080*0.05), 0), (0, int(1920*0.05)),
    (int(1080*0.05), int(1920*0.05)), (int(1080*0.02), int(1920*0.03)),
    (int(1080*0.03), int(1920*0.01)), (0, int(1920*0.03)), (int(1080*0.04), int(1920*0.02)),
]

W, H, FPS = 1080, 1920, 30
SCENE_COUNT = 8 if DURATION >= 40 else 6 if DURATION >= 28 else 4
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
completed = []
failed = []

def make_clip(img_path, dur, out, idx=0):
    ox, oy = CROP_OFFSETS[idx % len(CROP_OFFSETS)]
    sw, sh = int(W*1.12), int(H*1.12)
    vf = f"scale={sw}:{sh}:force_original_aspect_ratio=increase,crop={W}:{H}:{ox}:{oy}"
    if Path(img_path).exists():
        ff('-loop','1','-i',str(img_path),'-vf',vf,
           '-t',str(dur),'-r',str(FPS),'-an',
           '-c:v','libx264','-preset','ultrafast','-crf','23','-pix_fmt','yuv420p',str(out))
    else:
        ff('-f','lavfi','-i',f'color=c=black:s={W}x{H}:r={FPS}',
           '-t',str(dur),'-c:v','libx264','-preset','ultrafast',str(out))

def make_one_video(idx, angle):
    vid_num = idx + 1
    safe_angle = re.sub(r'[\\/:*?"<>|\s]','_',angle)[:25]
    print(f'\n{"─"*50}')
    print(f'🎬 [{vid_num}/{VIDEO_COUNT}] {angle}')
    print(f'{"─"*50}')
    TMP = Path(tempfile.mkdtemp(prefix=f'yt{vid_num}_'))
    IMG_TMP = TMP/'images'; IMG_TMP.mkdir()

    # ── 台本生成 ─────────────────────────────────────────────
    print('  📝 台本生成中...')
    labels = ['フック','共感','問題','解決①','解決②','解決③','まとめ','CTA'][:SCENE_COUNT]
    sc_str = '\n'.join([f'シーン{i+1}（{labels[i]}）: （セリフ）' for i in range(SCENE_COUNT)])
    kw_str = '\n'.join([f'scene{i+1}: （英語1〜2語）' for i in range(SCENE_COUNT)])
    raw = call_ai(
        f'YouTube Shorts台本ライターとして作成してください。\n'
        f'【タイトル】{angle}\n【テーマ】{THEME}\n【尺】{DURATION}秒/{SCENE_COUNT}シーン\n\n'
        f'【ルール】\n'
        f'・各セリフは10〜20文字の短い一文のみ\n'
        f'・シーン1は視聴者を掴む衝撃的な一言\n'
        f'・話し言葉（〜だよ、〜なんです、〜ね）\n'
        f'・テンポよく読めるリズム\n\n'
        f'[台本]\n{sc_str}\n[キーワード]\n{kw_str}'
    )

    script_text = (re.search(r'\[台本\]([\s\S]*?)(?=\[キーワード\])', raw) or
                   type('x',(),{'group':lambda s,n:raw})()).group(1)
    kw_block = re.search(r'\[キーワード\]([\s\S]*?)$', raw)
    keywords = []
    if kw_block:
        for line in kw_block.group(1).split('\n'):
            m = re.match(r'scene\d+[:\s]+(.+)', line.strip(), re.I)
            if m: keywords.append(m.group(1).strip())
    while len(keywords) < SCENE_COUNT: keywords.append('lifestyle')
    scene_lines = re.split(r'シーン\d+[（(][^)）]*[)）]\s*[:：]', script_text)
    scene_texts = [clean_text(s) for s in scene_lines[1:] if s.strip()]
    while len(scene_texts) < SCENE_COUNT: scene_texts.append(THEME)
    print('  ✓ 台本完了')

    # ── 画像取得 ──────────────────────────────────────────────
    print('  🖼 画像取得中...')
    scenes = []
    for i, kw in enumerate(keywords[:SCENE_COUNT]):
        time.sleep(0.4)
        fn = IMG_TMP/f'scene_{i+1:02d}.jpg'
        try:
            r = _req.get('https://api.pexels.com/v1/search',
                         headers={'Authorization': PEXELS_API_KEY},
                         params={'query': kw, 'per_page': 3, 'orientation': 'portrait'}, timeout=15)
            if r.status_code == 200:
                photos = r.json().get('photos', [])
                if photos:
                    photo = photos[i % len(photos)]
                    url = photo['src'].get('portrait') or photo['src'].get('large')
                    dl = _req.get(url, timeout=30)
                    if dl.status_code == 200: fn.write_bytes(dl.content)
        except Exception as e:
            print(f'  ⚠ 画像スキップ(scene{i+1}): {e}')
        scenes.append({'scene':i+1,'image':fn,'keyword':kw,
                       'text': scene_texts[i] if i < len(scene_texts) else THEME})
    print('  ✓ 画像完了')

    # ── スタイル変換 ──────────────────────────────────────────
    if IMAGE_STYLE in STYLES:
        fn_style = STYLES[IMAGE_STYLE]
        for s in scenes:
            if s['image'].exists():
                img = cv2.imread(str(s['image']))
                if img is not None: cv2.imwrite(str(s['image']), fn_style(img))
        print('  ✓ スタイル変換完了')

    # ── 音声生成（1.25倍速でテンポアップ） ───────────────────
    print('  🎙 音声生成中（1.25倍速）...')
    wavs = []
    for i, s in enumerate(scenes):
        txt = s['text'] or THEME
        mp3 = TMP/f'v{i}.mp3'
        wav = TMP/f'v{i}.wav'
        try:
            gTTS(text=txt, lang='ja', slow=False).save(str(mp3))
            # 1.25倍速に変換してテンポアップ
            ff('-i',str(mp3),
               '-filter:a','atempo=1.25',
               '-ar','44100','-ac','1',str(wav))
            audio_dur = get_wav_dur(wav)
            s['duration'] = max(audio_dur + 0.1, 2.0)  # バッファ0.1秒（最小限）
            wavs.append(wav)
        except Exception as e:
            s['duration'] = max(DURATION / SCENE_COUNT, 2.0)
            print(f'  ⚠ 音声スキップ(scene{i+1}): {e}')

    # 音声が一つもない場合はサイン波のBGMを生成
    if not wavs:
        print('  ⚠ 音声生成失敗。BGM音を生成します...')
        bgm_wav = TMP/'bgm.wav'
        total_dur = sum(s['duration'] for s in scenes)
        sr = 44100
        t_arr = np.linspace(0, total_dur, int(sr * total_dur))
        bgm = (np.sin(2*np.pi*220*t_arr)*0.2).astype(np.float32)
        import scipy.io.wavfile as wio
        wio.write(str(bgm_wav), sr, bgm)
        wavs.append(bgm_wav)

    print(f'  ✓ 音声完了（{len(wavs)}シーン）')

    # ── 動画クリップ生成 ──────────────────────────────────────
    print('  🎬 動画生成中...')
    clips = []
    for s in scenes:
        out = TMP/f"c{s['scene']:02d}.mp4"
        make_clip(s['image'], s['duration'], out, idx=s['scene']-1)
        clips.append(out)

    lf2 = TMP/'cl.txt'
    lf2.write_text('\n'.join(f"file '{p}'" for p in clips))
    mg = TMP/'m.mp4'
    ff('-f','concat','-safe','0','-i',str(lf2),'-c','copy',str(mg))

    # ── 字幕（実際のセリフを表示） ────────────────────────────
    ah = (
        f"[Script Info]\nPlayResX:{W}\nPlayResY:{H}\nScriptType:v4.00+\n\n"
        f"[V4+ Styles]\n"
        f"Format:Name,Fontname,Fontsize,PrimaryColour,SecondaryColour,OutlineColour,BackColour,"
        f"Bold,Italic,Underline,StrikeOut,ScaleX,ScaleY,Spacing,Angle,BorderStyle,Outline,Shadow,"
        f"Alignment,MarginL,MarginR,MarginV,Encoding\n"
        f"Style:Default,Noto Sans CJK JP,82,&H00FFFFFF,&H000000FF,&H00000000,&H00000000,"
        f"-1,0,0,0,100,100,0,0,1,5,2,2,30,30,{int(H*0.12)},1\n\n"
        f"[Events]\nFormat:Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n"
    )
    ev = []; t = 0.0
    for s in scenes:
        sub_text = wrap_subtitle(s['text'])
        ev.append(f"Dialogue:0,{at(t)},{at(t+s['duration'])},Default,,0,0,0,,{sub_text}")
        t += s['duration']
    sub = TMP/'s.ass'
    sub.write_text(ah+'\n'.join(ev), encoding='utf-8')
    se = str(sub).replace('\\','/').replace(':','\\:')

    # ── 音声を結合 ────────────────────────────────────────────
    lf = TMP/'vl.txt'
    lf.write_text('\n'.join(f"file '{p}'" for p in wavs))
    comb = TMP/'vc.wav'
    vaac = TMP/'v.aac'
    ff('-f','concat','-safe','0','-i',str(lf),'-c','copy',str(comb))
    ff('-i',str(comb),'-c:a','aac','-ar','44100',str(vaac))
    print(f'  ✓ 音声結合完了 ({vaac.stat().st_size//1024}KB)')

    # ── 最終合成 ──────────────────────────────────────────────
    safe_theme_f = re.sub(r'[\\/:*?"<>|\s]','_',THEME)[:15]
    OUT = OUTPUT_DIR/f'{vid_num:02d}_{safe_theme_f}_{safe_angle}.mp4'
    total_dur = sum(s['duration'] for s in scenes)

    ff('-i',str(mg),'-i',str(vaac),
       '-vf',f'ass={se}',
       '-map','0:v','-map','1:a',
       '-c:v','libx264','-preset','fast','-crf','20',
       '-c:a','aac','-b:a','192k',
       '-pix_fmt','yuv420p','-movflags','+faststart',
       '-t',str(min(total_dur, DURATION+5)),str(OUT))

    if not OUT.exists() or OUT.stat().st_size < 10000:
        raise RuntimeError(f'動画生成失敗: {OUT}')

    mb = OUT.stat().st_size/1_048_576
    print(f'  ✅ 完成！ {OUT.name} ({mb:.1f}MB, {total_dur:.0f}秒)')
    return str(OUT)

start_all = time.time()
for idx, angle in enumerate(ANGLES):
    try:
        out_path = make_one_video(idx, angle)
        completed.append((idx+1, angle, out_path))
    except Exception as e:
        import traceback
        print(f'\n  ❌ エラー: {traceback.format_exc()[-500:]}')
        failed.append((idx+1, angle, str(e)))

elapsed = (time.time()-start_all)/60
print(f'\n{"="*50}')
print(f'🎉 完了！ 成功:{len(completed)}本 / 失敗:{len(failed)}本 / {elapsed:.1f}分')
print(f'{"="*50}')

In [ ]:
# 【自動】完成動画をダウンロード（触らなくてOK）
from IPython.display import Video, display, Audio
from google.colab import files
import shutil, numpy as np

print(f'✅ 完成した動画 ({len(completed)}本):')
for num, angle, path in completed:
    mb = Path(path).stat().st_size / 1_048_576
    print(f'  {num:2d}. {angle}  [{mb:.1f}MB]')

if failed:
    print(f'\n❌ 失敗 ({len(failed)}本):')
    for num, angle, err in failed:
        print(f'  {num:2d}. {angle} → {err[:80]}')

# プレビュー表示
if completed:
    print('\n▶ 1本目をプレビュー:')
    shutil.copy(completed[0][2], '/content/preview.mp4')
    display(Video('/content/preview.mp4', width=360))

# 完了通知音
sr = 44100
beep = np.sin(2*np.pi*880*np.linspace(0,0.3,int(sr*0.3)))*0.5
silent = np.zeros(int(sr*0.1))
display(Audio(np.concatenate([beep,silent,beep]), rate=sr, autoplay=True))

# ダウンロード
print('\n⬇️ 動画をダウンロードしています...')
for num, angle, path in completed:
    print(f'  ダウンロード中: {Path(path).name}')
    files.download(path)